In [ ]:
# проверяем, что видеокарта доступна
import subprocess
from pathlib import Path

result = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
print(result.stdout if result.returncode == 0 else result.stderr)
if result.returncode != 0:
    raise RuntimeError("Для воспроизведения решения нужен CUDA GPU.")

# считаем текущую папку корнем проекта
PROJECT_ROOT = Path.cwd().resolve()
print("Корневая папка проекта:", PROJECT_ROOT)


In [ ]:
# ставим все зависимости из одного файла
import subprocess
import sys

requirements_path = PROJECT_ROOT / "requirements.txt"
if not requirements_path.exists():
    raise FileNotFoundError(
        f"Не найден файл с зависимостями: {requirements_path}"
    )

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_path),
    ]
)


In [ ]:
# задаем пути и настройки лучшего validation варианта
from pathlib import Path
import shutil
import tempfile

PROJECT_DIR = PROJECT_ROOT
# здесь лежат исходные данные
DATA_DIR = PROJECT_DIR / "datasets"
ARTIFACTS_DIR = PROJECT_DIR / "dense_artifacts"
MODELS_DIR = PROJECT_DIR / "models"
LOCAL_DATA_DIR = Path(tempfile.gettempdir()) / "avito_finetuned_answer"

BASE_MODEL_NAME = "intfloat/multilingual-e5-small"
FINETUNED_MODEL_NAME = "multilingual_e5_small_avito_v1"
FINETUNED_MODEL_DIR = MODELS_DIR / FINETUNED_MODEL_NAME
MODEL_COMPLETE_PATH = FINETUNED_MODEL_DIR / "training_complete.json"

# эти параметры должны совпадать с обучением модели
MAX_SEQ_LENGTH = 256
ENCODING_BATCH_SIZE = 128
SEARCH_BATCH_SIZE = 64
USE_FP16 = True
TOP_K = 50

# столько кандидатов берем из каждого поискового канала
DENSE_LOCAL_QUOTA = 32
BM25_LOCAL_QUOTA = 14
BM25_GLOBAL_QUOTA = 4
DENSE_GLOBAL_QUOTA = 0
DENSE_GLOBAL_DEPTH = 100
DENSE_LOCAL_DEPTH = 50
BM25_GLOBAL_DEPTH = 100
BM25_LOCAL_DEPTH = 50

SOURCE_ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"
SOURCE_QUERIES_PATH = DATA_DIR / "benchmark_queries.parquet"
LOCAL_ITEMS_PATH = LOCAL_DATA_DIR / "benchmark_items.parquet"
LOCAL_QUERIES_PATH = LOCAL_DATA_DIR / "benchmark_queries.parquet"
RUN_NAME = f"{FINETUNED_MODEL_NAME}_len{MAX_SEQ_LENGTH}"
EMBEDDINGS_PATH = ARTIFACTS_DIR / f"{RUN_NAME}_item_embeddings.npy"
PROGRESS_PATH = ARTIFACTS_DIR / f"{RUN_NAME}_progress.json"
FINAL_ANSWER_PATH = PROJECT_DIR / "answer.csv"
# до запуска поиска проверяем все нужные файлы и артефакты
required_paths = [
    SOURCE_ITEMS_PATH,
    SOURCE_QUERIES_PATH,
    FINETUNED_MODEL_DIR,
    MODEL_COMPLETE_PATH,
    EMBEDDINGS_PATH,
    PROGRESS_PATH,
]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
# копируем parquet во временную папку для более быстрого чтения
for source, destination in (
    (SOURCE_ITEMS_PATH, LOCAL_ITEMS_PATH),
    (SOURCE_QUERIES_PATH, LOCAL_QUERIES_PATH),
):
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
        print(f"Скопирован {source.name}: {destination.stat().st_size / 1024**2:.1f} МБ")
    else:
        print(f"Уже скопирован: {destination}")


In [ ]:
# загружаем benchmark и проверяем готовые артефакты
import hashlib
import json
import numpy as np
import pandas as pd

# читаем только нужные для поиска колонки
items = pd.read_parquet(
    LOCAL_ITEMS_PATH,
    columns=[
        "item_id",
        "item_location_id",
        "item_title_raw",
        "item_infm_params_text",
        "item_description_raw",
    ],
)
queries = pd.read_parquet(LOCAL_QUERIES_PATH)
items["item_id"] = items["item_id"].astype(str)
queries["query_id"] = queries["query_id"].astype(str)

if len(items) != 189_212 or items["item_id"].duplicated().any():
    raise ValueError("Некорректный benchmark_items.parquet")
if len(queries) != 2_452 or queries["query_id"].duplicated().any():
    raise ValueError("Некорректный benchmark_queries.parquet")

item_ids = items["item_id"].tolist()
query_ids = queries["query_id"].tolist()
item_ids_hash = hashlib.sha256("\n".join(item_ids).encode("utf-8")).hexdigest()
progress = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
training_metadata = json.loads(MODEL_COMPLETE_PATH.read_text(encoding="utf-8"))

# проверяем, что эмбеддинги построены этой моделью для этого корпуса
expected_progress_values = {
    "model_name": FINETUNED_MODEL_NAME,
    "base_model_name": BASE_MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "item_count": len(item_ids),
    "item_ids_sha256": item_ids_hash,
    "completed_items": len(item_ids),
}
for key, expected_value in expected_progress_values.items():
    if progress.get(key) != expected_value:
        raise ValueError(f"Артефакт эмбеддингов не прошёл проверку: {key}")

if progress.get("training_pairs_sha256") != training_metadata.get(
    "training_pairs_sha256"
):
    raise ValueError("Модель и эмбеддинги получены на разных обучающих парах")
if training_metadata.get("base_model_name") != BASE_MODEL_NAME:
    raise ValueError("Неожиданная базовая модель")
if training_metadata.get("max_seq_length") != MAX_SEQ_LENGTH:
    raise ValueError("Неожиданная максимальная длина модели")

# читаем большую матрицу с диска без полной загрузки в ram
item_embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
if item_embeddings.shape != (len(items), 384):
    raise ValueError(
        f"Некорректная форма fine-tuned эмбеддингов: {item_embeddings.shape}"
    )

# приводим запросы и документы к одному виду
def normalize_text(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

normalized_queries = normalize_text(queries["search_query"])
dense_query_texts = ("query: " + normalized_queries).tolist()

# убираем названия полей и оставляем только значения фильтров
filter_field_labels = [
    "срочная услуга (мультистатус)",
    "предмет или специальность",
    "кто оказывает услуги",
    "чем вы занимаетесь",
    "участие в пилоте cpl",
    "поиск по слотам",
    "рабочие дни",
    "ваши клиенты",
    "преподаватель",
    "занятия",
    "тип услуги",
    "вид услуги",
]
clean_filter_values = normalize_text(queries["search_infm_params_text"])
for field_label in sorted(filter_field_labels, key=len, reverse=True):
    clean_filter_values = clean_filter_values.str.replace(
        field_label,
        " ",
        regex=False,
    )
clean_filter_values = (
    clean_filter_values
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
# добавляем очищенные фильтры только в запрос для bm25
bm25_query_texts = [
    query if not filter_text else f"{query} {filter_text}"
    for query, filter_text in zip(
        normalized_queries,
        clean_filter_values,
        strict=True,
    )
]

# сохраняем локации отдельными массивами для быстрого сравнения
query_locations = queries["search_location_id"].to_numpy(dtype=np.int64)
item_locations = items["item_location_id"].to_numpy(dtype=np.int64)

print("Fine-tuned эмбеддинги объявлений:", item_embeddings.shape)
print("Benchmark-запросов:", len(query_ids))
print("Запросов с непустым очищенным фильтром:", int((clean_filter_values != "").sum()))
print("Пример dense-запроса:", dense_query_texts[0])
print("Пример BM25-запроса:", bm25_query_texts[0])


In [ ]:
# загружаем дообученную модель и кодируем все запросы
import torch
from sentence_transformers import SentenceTransformer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA недоступна. Переключите Colab на GPU runtime.")

DEVICE = "cuda"
# берем локально сохраненную модель, а не базовую модель из интернета
model = SentenceTransformer(str(FINETUNED_MODEL_DIR), device=DEVICE)
model.max_seq_length = MAX_SEQ_LENGTH
if USE_FP16:
    model.half()

# нормализуем эмбеддинги, чтобы скалярное произведение стало cosine similarity
query_embeddings = model.encode(
    dense_query_texts,
    batch_size=ENCODING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    device=DEVICE,
).astype(np.float32)

if query_embeddings.shape != (len(query_ids), item_embeddings.shape[1]):
    raise ValueError("Размерность эмбеддингов запросов и корпуса не совпадает")

print("GPU:", torch.cuda.get_device_name(0))
print("Fine-tuned эмбеддинги запросов:", query_embeddings.shape)


In [ ]:
# получаем глобальную и локальную dense выдачу
import gc
from tqdm.auto import tqdm

# переносим эмбеддинги на gpu для точного поиска по всему корпусу
corpus_tensor = torch.from_numpy(np.asarray(item_embeddings)).to(DEVICE)
query_tensor = torch.from_numpy(query_embeddings).to(DEVICE)
item_locations_tensor = torch.from_numpy(item_locations).to(DEVICE)
dense_global_rankings = []
dense_local_rankings = []

# градиенты здесь не нужны, поэтому поиск занимает меньше памяти
with torch.inference_mode():
    for start in tqdm(
        range(0, len(query_ids), SEARCH_BATCH_SIZE),
        desc="Fine-tuned dense rankings",
    ):
        end = min(start + SEARCH_BATCH_SIZE, len(query_ids))
        scores = query_tensor[start:end] @ corpus_tensor.T
        global_indices = torch.topk(
            scores,
            k=DENSE_GLOBAL_DEPTH,
            dim=1,
            largest=True,
            sorted=True,
        ).indices.cpu().numpy()

        # для локальной выдачи скрываем объявления из других локаций
        batch_locations = torch.from_numpy(query_locations[start:end]).to(DEVICE)
        same_location = batch_locations[:, None].eq(item_locations_tensor[None, :])
        local_scores = scores.masked_fill(~same_location, float("-inf"))
        local_indices = torch.topk(
            local_scores,
            k=DENSE_LOCAL_DEPTH,
            dim=1,
            largest=True,
            sorted=True,
        ).indices.cpu().numpy()

        for row_number in range(end - start):
            query_location = query_locations[start + row_number]
            dense_global_rankings.append(global_indices[row_number].tolist())
            dense_local_rankings.append(
                [
                    index
                    for index in local_indices[row_number].tolist()
                    if item_locations[index] == query_location
                ]
            )

        del scores, local_scores, same_location

print("Глобальные fine-tuned dense rankings готовы:", len(dense_global_rankings))
print("Локальные fine-tuned dense rankings готовы:", len(dense_local_rankings))

# после поиска освобождаем память перед построением bm25
del query_embeddings
del corpus_tensor
del query_tensor
del item_locations_tensor
del item_embeddings
del model
gc.collect()
torch.cuda.empty_cache()
print("Dense-модель и матрицы удалены из RAM")


In [ ]:
# строим bm25 с русским стеммингом и очищенными фильтрами
import bm25s
import Stemmer
from bm25s.tokenization import Tokenizer

# заголовок повторяем три раза, чтобы он сильнее влиял на поиск
titles = normalize_text(items["item_title_raw"])
params = normalize_text(items["item_infm_params_text"]).str.slice(0, 700)
descriptions = normalize_text(items["item_description_raw"]).str.slice(0, 800)
bm25_corpus_texts = (
    titles + " " + titles + " " + titles + " " + params + " " + descriptions
).tolist()

# разбиваем текст на токены и приводим русские слова к общей основе
stemmer = Stemmer.Stemmer("russian")
tokenizer = Tokenizer(lower=True, stopwords="russian", stemmer=stemmer)
corpus_tokens = tokenizer.tokenize(
    bm25_corpus_texts,
    update_vocab=True,
    return_as="tuple",
    show_progress=True,
)
retriever = bm25s.BM25(k1=1.5, b=0.75, method="lucene")
retriever.index(corpus_tokens, show_progress=True)
query_tokens = tokenizer.tokenize(
    bm25_query_texts,
    update_vocab=False,
    return_as="tuple",
    allow_empty=False,
    show_progress=True,
)

# заранее собираем объявления каждой локации
location_to_indices = {}
for document_index, location_id in enumerate(item_locations):
    location_to_indices.setdefault(int(location_id), []).append(document_index)
location_to_indices = {
    location_id: np.asarray(indices, dtype=np.int64)
    for location_id, indices in location_to_indices.items()
}

def top_positive_indices(
    scores: np.ndarray,
    k: int,
    candidates: np.ndarray | None = None,
) -> list[int]:
    # документы с нулевой оценкой не берем, свободные места заполнит dense
    if candidates is None:
        eligible = np.flatnonzero(scores > 0)
        eligible_scores = scores[eligible]
    else:
        candidate_scores = scores[candidates]
        positive_mask = candidate_scores > 0
        eligible = candidates[positive_mask]
        eligible_scores = candidate_scores[positive_mask]

    if len(eligible) == 0:
        return []
    if len(eligible) > k:
        selected = np.argpartition(eligible_scores, -k)[-k:]
        selected = selected[np.argsort(eligible_scores[selected])[::-1]]
    else:
        selected = np.argsort(eligible_scores)[::-1]
    return eligible[selected].astype(np.int64).tolist()

# для каждого запроса сохраняем глобальную и локальную выдачу
bm25_global_rankings = []
bm25_local_rankings = []
for row_number, query_token_ids in enumerate(
    tqdm(query_tokens.ids, desc="BM25 rankings с clean filters")
):
    scores = retriever.get_scores_from_ids(query_token_ids)
    bm25_global_rankings.append(
        top_positive_indices(scores, BM25_GLOBAL_DEPTH)
    )
    local_candidates = location_to_indices.get(
        int(query_locations[row_number]),
        np.empty(0, dtype=np.int64),
    )
    bm25_local_rankings.append(
        top_positive_indices(scores, BM25_LOCAL_DEPTH, local_candidates)
    )

print("Глобальные BM25 rankings готовы:", len(bm25_global_rankings))
print("Локальные BM25 rankings готовы:", len(bm25_local_rankings))

# удаляем индекс и тексты, потому что дальше нужны только готовые списки
del titles
del params
del descriptions
del bm25_corpus_texts
del corpus_tokens
del query_tokens
del retriever
gc.collect()
print("Временные тексты и индекс BM25 удалены из RAM")


In [ ]:
# объединяем каналы по квотам и не допускаем повторов
# функция добавляет только новые объявления и соблюдает квоту
def add_unique_with_quota(
    result: list[int],
    candidates: list[int],
    quota: int,
) -> int:
    if quota <= 0:
        return 0
    added = 0
    seen = set(result)
    for document_index in candidates:
        if document_index not in seen:
            result.append(document_index)
            seen.add(document_index)
            added += 1
        if added == quota:
            break
    return added

predicted_indices = []
channel_counts = {
    "dense_local": [],
    "bm25_local": [],
    "bm25_global": [],
    "dense_global": [],
    "dense_global_fallback": [],
}

# собираем ровно 50 кандидатов для каждого запроса
for row_number, query_id in enumerate(query_ids):
    result = []

    channel_counts["dense_local"].append(
        add_unique_with_quota(
            result,
            dense_local_rankings[row_number],
            DENSE_LOCAL_QUOTA,
        )
    )
    channel_counts["bm25_local"].append(
        add_unique_with_quota(
            result,
            bm25_local_rankings[row_number],
            BM25_LOCAL_QUOTA,
        )
    )
    channel_counts["bm25_global"].append(
        add_unique_with_quota(
            result,
            bm25_global_rankings[row_number],
            BM25_GLOBAL_QUOTA,
        )
    )
    channel_counts["dense_global"].append(
        add_unique_with_quota(
            result,
            dense_global_rankings[row_number],
            DENSE_GLOBAL_QUOTA,
        )
    )

    # если после основных каналов мест не хватило, добираем global dense
    fallback_count = 0
    if len(result) < TOP_K:
        fallback_count = add_unique_with_quota(
            result,
            dense_global_rankings[row_number],
            TOP_K - len(result),
        )
    channel_counts["dense_global_fallback"].append(fallback_count)

    if len(result) != TOP_K or len(result) != len(set(result)):
        raise ValueError(
            f"Для запроса {query_id} получено {len(result)} уникальных кандидатов"
        )

    predicted_indices.append(result)

# превращаем номера строк корпуса обратно в item id
predictions = [
    [item_ids[index] for index in row]
    for row in predicted_indices
]

print("Запросов обработано:", len(predictions))
for channel_name, counts in channel_counts.items():
    print(f"Среднее число {channel_name}: {np.mean(counts):.3f}")


In [ ]:
# создаем answer csv и проверяем его перед сохранением
import re

# в итоговом файле должны быть только две колонки
answer = pd.DataFrame(
    {
        "query_id": query_ids,
        "answer": [" ".join(row) for row in predictions],
    }
)

# проверяем число строк, названия колонок и все query id
if answer.columns.tolist() != ["query_id", "answer"]:
    raise ValueError("В answer должны быть ровно две колонки")
if len(answer) != 2_452 or answer["query_id"].nunique() != 2_452:
    raise ValueError("Некорректное число или повторы query_id")
if set(answer["query_id"]) != set(queries["query_id"]):
    raise ValueError("Набор query_id не совпадает с benchmark_queries")
if not answer["query_id"].str.fullmatch(r".{16}").all():
    raise ValueError("Некоторые query_id имеют длину не 16 символов")

# проверяем каждый item id и количество кандидатов
corpus_item_ids = set(item_ids)
item_id_pattern = re.compile(r"^[0-9a-f]{16}$")
for query_id, row in zip(query_ids, predictions, strict=True):
    if len(row) != TOP_K or len(row) != len(set(row)):
        raise ValueError(f"Некорректные кандидаты для {query_id}")
    if any(item_id not in corpus_item_ids for item_id in row):
        raise ValueError(f"Неизвестный item_id для {query_id}")
    if any(item_id_pattern.fullmatch(item_id) is None for item_id in row):
        raise ValueError(f"Некорректный формат item_id для {query_id}")

# сохраняем csv без индекса и дополнительных колонок
answer.to_csv(FINAL_ANSWER_PATH, index=False, encoding="utf-8")

print("Готов основной файл:", FINAL_ANSWER_PATH)
print("Строк:", len(answer))
print("Кандидатов на запрос:", TOP_K)
display(answer.head())
